변경 o

# Baseline

In [1]:
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

Read dataset files

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 18 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   case_id                            300000 non-null  int64  
 1   Hospital                           300000 non-null  int64  
 2   Hospital_type                      300000 non-null  int64  
 3   Hospital_city                      300000 non-null  int64  
 4   Hospital_region                    300000 non-null  int64  
 5   Available_Extra_Rooms_in_Hospital  300000 non-null  int64  
 6   Department                         300000 non-null  object 
 7   Ward_Type                          300000 non-null  object 
 8   Ward_Facility                      300000 non-null  object 
 9   Bed_Grade                          299897 non-null  float64
 10  patientid                          300000 non-null  int64  
 11  City_Code_Patient                  2957

In [4]:
train_df.Class.value_counts()

0    178399
1     95881
2     25720
Name: Class, dtype: int64

Make dataset

In [5]:
ordinal_cols = ['Department', 'Ward_Type', 'Ward_Facility', 'Type of Admission', 'Illness_Severity', 'Age']

for i in ordinal_cols:
    train_df[i] = train_df[i].astype('category').cat.codes
    test_df[i] = test_df[i].astype('category').cat.codes
X_train = train_df.iloc[:, :-1]
y_train = train_df.iloc[:, -1]
X_test = test_df

train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 18 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   case_id                            300000 non-null  int64  
 1   Hospital                           300000 non-null  int64  
 2   Hospital_type                      300000 non-null  int64  
 3   Hospital_city                      300000 non-null  int64  
 4   Hospital_region                    300000 non-null  int64  
 5   Available_Extra_Rooms_in_Hospital  300000 non-null  int64  
 6   Department                         300000 non-null  int8   
 7   Ward_Type                          300000 non-null  int8   
 8   Ward_Facility                      300000 non-null  int8   
 9   Bed_Grade                          299897 non-null  float64
 10  patientid                          300000 non-null  int64  
 11  City_Code_Patient                  2957

In [6]:
train_df.head()

,case_id,Hospital,Hospital_type,Hospital_city,Hospital_region,Available_Extra_Rooms_in_Hospital,Department,Ward_Type,Ward_Facility,Bed_Grade,patientid,City_Code_Patient,Type of Admission,Illness_Severity,Patient_Visitors,Age,Admission_Deposit,Class
0,0,12,0,9,1,4,2,1,1,4.0,10295,7.0,1,1,4,0,7589.0,0
1,1,6,0,6,0,3,2,1,5,1.0,101514,15.0,1,2,3,6,5625.0,1
2,2,18,3,13,1,2,1,2,1,4.0,79233,23.0,1,0,2,5,4421.0,0
3,3,19,0,7,1,4,3,1,2,1.0,23871,8.0,0,2,4,5,4003.0,0
4,4,30,2,3,2,3,2,1,0,2.0,125694,1.0,0,2,4,4,3864.0,0


In [7]:
from sklearn.model_selection import train_test_split

X = pd.concat([X_train, X_test], axis=0)

X_train = X.iloc[:len(X_train)]
X_test = X.iloc[len(X_train):]
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=1)

In [8]:
X_train.head()

,case_id,Hospital,Hospital_type,Hospital_city,Hospital_region,Available_Extra_Rooms_in_Hospital,Department,Ward_Type,Ward_Facility,Bed_Grade,patientid,City_Code_Patient,Type of Admission,Illness_Severity,Patient_Visitors,Age,Admission_Deposit
128309,128309,20,1,2,1,3,3,3,3,4.0,110673,4.0,0,1,2,3,3380.0
197694,197694,24,0,1,0,3,2,2,4,3.0,18144,10.0,1,1,2,3,4993.0
199798,199798,10,4,1,0,6,3,1,4,2.0,123042,2.0,0,0,2,2,3536.0
229590,229590,19,0,7,1,4,3,1,2,3.0,3852,9.0,0,0,3,4,4345.0
197764,197764,10,4,1,0,3,0,3,4,2.0,83564,15.0,1,2,4,3,4981.0


In [9]:
from sklearn.model_selection import RandomizedSearchCV

params = {
          'n_estimators': [50, 100, 200],
          'learning_rate': [0.01, 0.05, 0.1, 0.3], 
          'max_depth': [5,10,15,20],          
          'subsample': [0.6, 0.8, 1],
          'min_child_weight': [ 1, 3, 5, 7 ],
          'gamma': [0.5, 1, 1.5, 2,3,4, 5],
          'colsample_bytree': [ 0.3, 0.4, 0.5 , 0.7, 1.0 ]
            }

estimator = xgb.XGBClassifier(device='gpu')

random_search = RandomizedSearchCV(estimator=estimator,
                          param_distributions=params,
                          cv=5,
                          scoring='f1_weighted',
                          n_jobs=-1)

random_search.fit(X_train, y_train)
random_search.best_params_

{'subsample': 0.6,
 'n_estimators': 100,
 'min_child_weight': 1,
 'max_depth': 15,
 'learning_rate': 0.3,
 'gamma': 2,
 'colsample_bytree': 1.0}

In [10]:
val_pred = random_search.best_estimator_.predict(X_val)
print(classification_report(y_val, val_pred))

              precision    recall  f1-score   support

           0       0.77      0.94      0.85     17854
           1       0.68      0.49      0.57      9603
           2       0.68      0.36      0.47      2543

    accuracy                           0.74     30000
   macro avg       0.71      0.60      0.63     30000
weighted avg       0.73      0.74      0.72     30000



C:\Users\tubs5\anaconda3\lib\site-packages\xgboost\core.py:160: UserWarning: [23:16:12] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0750514818a16474a-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Make model & train

In [11]:
model = xgb.XGBClassifier(device='gpu')
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print(classification_report(y_val, val_pred))

              precision    recall  f1-score   support

           0       0.77      0.95      0.85     17854
           1       0.69      0.48      0.56      9603
           2       0.72      0.35      0.47      2543

    accuracy                           0.75     30000
   macro avg       0.73      0.59      0.63     30000
weighted avg       0.74      0.75      0.73     30000



Make test prediction

In [16]:
test_pred = random_search.best_estimator_.predict(X_test)

Export prediction to file

In [17]:
submission_df = pd.read_csv('sample_submission.csv')
submission_df.Class = test_pred
submission_df.to_csv('submission.csv', index=False)